# Autoencoder on plausibilitree-DL point clouds

Bare-bones PointNet-style **autoencoder** (PyTorch), unsupervised — it only
reads the feature files from `prepare.py` (labels are not used):

- `features/<plot_uid>.npz` — `points` (N, 3) float32 plus every extra point
  dimension (`semantic_code`, `tree_id`, `shrub_id`, `classification`,
  `branch_code`, `branch_id`, `height_above_ground`) and a `meta` JSON string

One plot = one sample. Pick the device (CPU / Apple GPU via MPS / CUDA), the
architecture (`ARCH`: `"pointnet"` or `"pointnet++"`), the feature directory,
the extra point channels to train on (`EXTRA_DIMS`), how many plots to load
(`N_SAMPLES`), the loss (`LOSS`), the model shape (`N_POINTS`, `LATENT_DIM`),
and the output range (`OUTPUT_RANGE`). Set `N_SAMPLES = 1` to load a single
sample and tweak the model until it overfits it (then there is no validation
split, so early stopping stays off). Optional early stopping via
`PATIENCE`/`MIN_DELTA` on the validation split. The last cell plots original
vs reconstructed points in the x-y, x-z, and y-z projections.

After training, the model (config + weights) is saved to `CKPT_PATH` so that
`predict_from_embeddings.ipynb` can reuse the encoder to predict labels.

Requires: `torch`, `numpy` (and `matplotlib` only for the optional plots).

In [ ]:
import glob
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

# ---- device: CPU / Apple integrated GPU (MPS) / CUDA cores ---------------
DEVICE_CHOICE = "auto"  # "auto" | "cpu" | "mps" | "cuda"


def pick_device(choice: str) -> torch.device:
    if choice in ("cuda", "auto") and torch.cuda.is_available():
        return torch.device("cuda")
    if choice in ("mps", "auto") and torch.backends.mps.is_available():
        return torch.device("mps")  # Apple integrated GPU
    if choice != "auto":
        print(f"warning: {choice!r} not available, falling back to CPU")
    return torch.device("cpu")


device = pick_device(DEVICE_CHOICE)
print("device:", device)

In [ ]:
# ---- user config -----------------------------------------------------------
FEATURES_DIR = "../dl_dataset/features"  # prepare.py -o <out>/features
EXTRA_DIMS = ["semantic_code"]           # extra per-point channels to train on,
                                         # stacked onto xyz (input = 3 + len(EXTRA_DIMS))
ARCH = "pointnet"    # "pointnet" | "pointnet++" (set abstraction + feature
                     # propagation; radii are in model space, see SA_RADII)
N_POINTS = 1024      # each cloud is padded/tiled or truncated to this many points
N_SAMPLES = 1        # how many plots to load: an int, or "all"; 1 = overfit one sample
LATENT_DIM = 128     # bottleneck size
LOSS = "mse"         # "mse" | "l1" | "huber" | "chamfer" (xyz Chamfer distance;
                     # note chamfer is O(N_POINTS^2) per sample)
OUTPUT_RANGE = "neg1_1"  # bound the decoder's xyz output to the normalized
                         # range: "neg1_1" -> tanh ([-1,1]), "zero_1" -> sigmoid
                         # ([0,1]), "none" -> linear. Match prepare.py's
                         # --normalize; extra dims are NOT bounded.
# PointNet++ sampling/grouping (only used when ARCH == "pointnet++"):
SA_POINTS = [512, 128]   # points kept per set-abstraction level (coarse last);
                         # every entry must be <= N_POINTS
SA_RADII = [0.2, 0.4]    # ball-query radius per level, in model space
                         # (same units as the normalized coords)
SA_NSAMPLE = 32          # max points grouped per centroid (nearest fill the rest)
CKPT_PATH = "../checkpoints/autoencoder.pt"  # where to save the trained model
                                             # (config + weights) after training;
                                             # None = do not save. Used by
                                             # predict_from_embeddings.ipynb
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-3
DROPOUT = 0.3    # dropout probability in the autoencoder (0 = off)
SEED = 42
VAL_FRAC = 0.2     # validation fraction (only used when >1 sample is loaded)
PATIENCE = None    # early stopping on the val loss: stop after this many
                   # epochs without improvement; None = off
MIN_DELTA = 0.0    # minimum val-loss improvement that resets the patience counter

_first = sorted(glob.glob(os.path.join(FEATURES_DIR, "*.npz")))
if not _first:
    raise FileNotFoundError(f"no feature files in {FEATURES_DIR!r} - run prepare.py first")
with np.load(_first[0]) as z:
    print("available feature arrays:", z.files)

In [ ]:
class CloudDataset(Dataset):
    """One plot per sample: a (C, N_POINTS) feature matrix from the .npz
    (points + EXTRA_DIMS channels). Optionally limited to the first
    n_samples plots."""

    def __init__(self, features_dir, extra_dims, n_points, n_samples=None):
        self.n_points = n_points
        self.extra_dims = list(extra_dims)
        feats = sorted(glob.glob(os.path.join(features_dir, "*.npz")))
        if n_samples is not None:
            feats = feats[:n_samples]
        if not feats:
            raise FileNotFoundError(f"no feature files in {features_dir!r}")
        self.samples = feats
        self.names = [os.path.splitext(os.path.basename(f))[0] for f in feats]
        with np.load(feats[0]) as z:
            available = set(z.files)
        missing = [k for k in self.extra_dims if k not in available]
        if missing:
            raise KeyError(f"feature dims {missing} not in the .npz; "
                           f"available: {sorted(available)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        with np.load(self.samples[i]) as z:
            cols = [z["points"].astype(np.float32)]          # (N, 3)
            for k in self.extra_dims:
                cols.append(z[k].astype(np.float32)[:, None])  # (N, 1)
        mat = np.concatenate(cols, axis=1)                   # (N, C)
        # fix the point count: tile if short, truncate if long
        n = len(mat)
        if n >= self.n_points:
            mat = mat[: self.n_points]
        else:
            reps = int(np.ceil(self.n_points / n))
            mat = np.tile(mat, (reps, 1))[: self.n_points]
        return torch.from_numpy(mat.T.copy())                 # (C, N_POINTS)


dataset = CloudDataset(FEATURES_DIR, EXTRA_DIMS, N_POINTS,
                       n_samples=None if N_SAMPLES == "all" else N_SAMPLES)
print(f"loaded {len(dataset)} sample(s): {dataset.names}")
print(f"input channels = {3 + len(EXTRA_DIMS)} (xyz + {EXTRA_DIMS})")

In [ ]:
def chamfer_loss(a, b):
    """Symmetric Chamfer distance on the xyz channels. a, b: (B, C, N).
    O(N^2) memory per sample - keep N_POINTS modest."""
    pa = a[:, :3].transpose(1, 2)  # (B, N, 3)
    pb = b[:, :3].transpose(1, 2)
    d = torch.cdist(pa, pb)        # (B, N, N)
    return (d.min(dim=2).values.mean(dim=1) + d.min(dim=1).values.mean(dim=1)).mean()


def bound_xyz(out, output_range):
    """Bound the decoder's xyz channels (first 3) to output_range; extra dims
    pass through linearly."""
    if output_range == "neg1_1":
        return torch.cat([torch.tanh(out[:, :3]), out[:, 3:]], dim=1)
    if output_range == "zero_1":
        return torch.cat([torch.sigmoid(out[:, :3]), out[:, 3:]], dim=1)
    return out


class PointNetAE(nn.Module):
    """Bare-bones PointNet autoencoder: shared point MLP -> global max pool ->
    FC encoder to a latent vector -> FC decoder back to (C, N)."""

    def __init__(self, in_channels, n_points, latent_dim, output_range="none"):
        super().__init__()
        self.in_channels = in_channels
        self.n_points = n_points
        self.output_range = output_range
        self.enc = nn.Sequential(
            nn.Conv1d(in_channels, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(DROPOUT),
        )
        self.enc_fc = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, n_points * in_channels),
        )

    def encode(self, x):  # (B, C, N) -> latent (B, L)
        return self.enc_fc(torch.max(self.enc(x), dim=2).values)

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        z = self.encode(x)
        out = self.dec_fc(z).view(-1, self.in_channels, self.n_points)
        return bound_xyz(out, self.output_range), z


# ---- PointNet++ building blocks ---------------------------------------------
def index_points(points, idx):
    """points: (B, N, C); idx: (B, S) or (B, S, G) -> (B, S, C) or (B, S, G, C)."""
    if idx.dim() == 2:
        idx = idx.unsqueeze(-1).expand(*idx.shape, points.shape[-1])
        return torch.gather(points, 1, idx)
    points = points.unsqueeze(1).expand(-1, idx.shape[1], -1, -1)
    idx = idx.unsqueeze(-1).expand(*idx.shape, points.shape[-1])
    return torch.gather(points, 2, idx)


def farthest_point_sample(xyz, npoint):
    """Iterative FPS. xyz: (B, N, 3) -> LongTensor (B, npoint).
    Python loop over npoint - fine for a few hundred samples."""
    B, N, _ = xyz.shape
    device = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=device)
    dist = torch.full((B, N), 1e10, device=device)
    farthest = torch.randint(0, N, (B,), device=device)
    batch = torch.arange(B, device=device)
    for i in range(npoint):
        idx[:, i] = farthest
        centroid = xyz[batch, farthest, :].unsqueeze(1)  # (B, 1, 3)
        d = ((xyz - centroid) ** 2).sum(-1)
        dist = torch.min(dist, d)
        farthest = dist.argmax(dim=1)  # (B,) farthest point per batch row
    return idx


def ball_query(radius, nsample, xyz, new_xyz):
    """Ball query: up to nsample points within radius of each centroid;
    centroids with fewer neighbors repeat their nearest point.
    xyz: (B, N, 3), new_xyz: (B, S, 3) -> idx (B, S, nsample)."""
    d = torch.cdist(new_xyz, xyz)                       # (B, S, N)
    order = d.argsort(dim=-1)                           # nearest first
    idx = order[:, :, :nsample]                         # (B, S, nsample)
    in_ball = (torch.gather(d, 2, idx) <= radius)       # (B, S, nsample)
    # replace out-of-ball picks with the nearest point (order[:, :, 0])
    first = order[:, :, :1].expand(-1, -1, nsample)
    return torch.where(in_ball, idx, first)


class SetAbstraction(nn.Module):
    """One PointNet++ set-abstraction level: FPS sample -> ball-query group ->
    shared point MLP on (relative xyz + features) -> max pool per group."""

    def __init__(self, npoint, radius, nsample, in_ch, mlp):
        super().__init__()
        self.npoint, self.radius, self.nsample = npoint, radius, nsample
        layers = []
        ch = in_ch + 3  # relative xyz is concatenated to the point features
        for w in mlp:
            layers += [nn.Conv2d(ch, w, 1), nn.BatchNorm2d(w), nn.ReLU(), nn.Dropout(DROPOUT)]
            ch = w
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, points):
        # xyz: (B, N, 3); points: (B, C, N)
        new_xyz = index_points(xyz, farthest_point_sample(xyz, self.npoint))  # (B, S, 3)
        idx = ball_query(self.radius, self.nsample, xyz, new_xyz)            # (B, S, G)
        grouped_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)           # (B, S, G, 3)
        grouped_feat = index_points(points.transpose(1, 2), idx)              # (B, S, G, C)
        new_points = torch.cat([grouped_xyz, grouped_feat], dim=-1)
        new_points = self.mlp(new_points.permute(0, 3, 2, 1))                 # (B, W, G, S)
        new_points = torch.max(new_points, dim=2).values                      # (B, W, S)
        return new_xyz, new_points


class FeaturePropagation(nn.Module):
    """One PointNet++ feature-propagation level: k-NN inverse-distance
    interpolation of the coarse features -> concat skip connection -> MLP."""

    def __init__(self, in_ch, mlp):
        super().__init__()
        layers = []
        ch = in_ch
        for w in mlp:
            layers += [nn.Conv1d(ch, w, 1), nn.BatchNorm1d(w), nn.ReLU(), nn.Dropout(DROPOUT)]
            ch = w
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, new_xyz, points, new_points):
        # xyz: (B, N, 3) target level; new_xyz: (B, M, 3) source level
        # points: (B, C1, N) skip features; new_points: (B, C2, M) coarse features
        M = new_xyz.shape[1]
        if M == 1:
            interp = new_points.expand(-1, -1, xyz.shape[1])
        else:
            k = min(3, M)                                    # 3-NN (fewer if M < 3)
            d = torch.cdist(new_xyz, xyz)                    # (B, M, N)
            dist, idx = d.topk(k, dim=1, largest=False)      # k-NN of each target
            weight = 1.0 / (dist + 1e-8)
            weight = weight / weight.sum(dim=1, keepdim=True)
            interp = (index_points(new_points.transpose(1, 2), idx)
                      * weight.unsqueeze(-1)).sum(dim=1)        # (B, N, C2)
            interp = interp.transpose(1, 2)                  # (B, C2, N)
        new_points = torch.cat([interp, points], dim=1) if points is not None else interp
        return self.mlp(new_points)


class PointNetPP_AE(nn.Module):
    """PointNet++ autoencoder: two set-abstraction levels -> global max pool ->
    FC latent -> FC back to the coarsest level -> feature-propagation decoder
    with skip connections -> per-point head. Decoder xyz output is bounded to
    `output_range`."""

    def __init__(self, in_channels, n_points, latent_dim, output_range="none",
                 sa_points=(512, 128), sa_radii=(0.2, 0.4), sa_nsample=32):
        super().__init__()
        if len(sa_points) != len(sa_radii):
            raise ValueError("sa_points and sa_radii must have the same length")
        if max(sa_points) > n_points:
            raise ValueError(f"max(SA_POINTS)={max(sa_points)} exceeds "
                             f"N_POINTS={n_points}")
        if output_range not in ("none", "neg1_1", "zero_1"):
            raise ValueError(f"output_range must be none|neg1_1|zero_1, got {output_range!r}")
        self.in_channels = in_channels
        self.n_points = n_points
        self.output_range = output_range
        c0, c1 = 128, 256
        self.sa1 = SetAbstraction(sa_points[0], sa_radii[0], sa_nsample,
                                  in_channels, [64, 64, c0])
        self.sa2 = SetAbstraction(sa_points[1], sa_radii[1], sa_nsample,
                                  c0, [128, 128, c1])
        self.enc_fc = nn.Sequential(
            nn.Linear(c1, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, c1 * sa_points[1]),
        )
        self.fp1 = FeaturePropagation(c1 + c0, [256, 256])
        self.fp2 = FeaturePropagation(256 + in_channels, [256, 128])
        self.head = nn.Sequential(
            nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Conv1d(128, in_channels, 1),
        )

    def encode(self, x):  # (B, C, N) -> latent (B, L)
        xyz = x[:, :3].transpose(1, 2)                    # (B, N, 3)
        l1_xyz, l1_points = self.sa1(xyz, x)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        return self.enc_fc(torch.max(l2_points, dim=2).values)

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        xyz = x[:, :3].transpose(1, 2)                    # (B, N, 3)
        l1_xyz, l1_points = self.sa1(xyz, x)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        z = self.enc_fc(torch.max(l2_points, dim=2).values)          # (B, L)
        coarse = self.dec_fc(z).view(-1, l2_points.shape[1], l2_xyz.shape[1])
        p1 = self.fp1(l1_xyz, l2_xyz, l1_points, coarse)             # (B, 256, S1)
        p0 = self.fp2(xyz, l1_xyz, x, p1)                            # (B, 128, N)
        return bound_xyz(self.head(p0), self.output_range), z


def build_model(arch, in_channels, n_points, latent_dim, output_range,
                sa_points, sa_radii, sa_nsample):
    if arch == "pointnet":
        return PointNetAE(in_channels, n_points, latent_dim, output_range)
    if arch in ("pointnet++", "pointnetpp"):
        return PointNetPP_AE(in_channels, n_points, latent_dim, output_range,
                             tuple(sa_points), tuple(sa_radii), sa_nsample)
    raise ValueError(f"unknown ARCH {arch!r} (pointnet | pointnet++)")


def make_criterion(name):
    if name == "mse":
        return nn.MSELoss()
    if name == "l1":
        return nn.L1Loss()
    if name == "huber":
        return nn.HuberLoss()
    if name == "chamfer":
        return chamfer_loss
    raise ValueError(f"unknown LOSS {name!r} (mse | l1 | huber | chamfer)")


model = build_model(ARCH, in_channels=3 + len(EXTRA_DIMS), n_points=N_POINTS,
                    latent_dim=LATENT_DIM, output_range=OUTPUT_RANGE,
                    sa_points=SA_POINTS, sa_radii=SA_RADII, sa_nsample=SA_NSAMPLE).to(device)
print(f"arch={ARCH}")
print(model)

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

n_val = max(1, int(round(len(dataset) * VAL_FRAC))) if len(dataset) > 1 else 0
n_train = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(SEED))
# drop_last avoids a final 1-sample batch, which BatchNorm1d rejects in train mode
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=len(train_set) > BATCH_SIZE)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE) if n_val else None

criterion = make_criterion(LOSS)
opt = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")
bad = 0
best_state = None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    mse_total = 0.0
    n = 0
    for x in train_loader:
        x = x.to(device)
        if x.shape[0] == 1:
            # single-sample overfit mode: BatchNorm needs >1 row per batch
            x = x.repeat(BATCH_SIZE, 1, 1)
        opt.zero_grad()
        recon, _ = model(x)
        loss = criterion(recon, x)
        loss.backward()
        opt.step()
        total += loss.item() * x.shape[0]
        mse_total += nn.functional.mse_loss(recon, x).item() * x.shape[0]
        n += x.shape[0]
    msg = f"epoch {epoch:3d}/{EPOCHS}  {LOSS} {total / max(n, 1):.6f}  MSE {mse_total / max(n, 1):.6f}"
    if val_loader is not None:
        model.eval()
        val_total = 0.0
        vn = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                recon, _ = model(x)
                val_total += criterion(recon, x).item() * x.shape[0]
                vn += x.shape[0]
        val_loss = val_total / max(vn, 1)
        msg += f"  val {LOSS} {val_loss:.6f}"
        if PATIENCE is not None:
            if val_loss < best_val - MIN_DELTA:
                best_val = val_loss
                bad = 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad > PATIENCE:
                    print(f"early stopping at epoch {epoch} "
                          f"(val {LOSS} no improvement for {PATIENCE} epochs)")
                    break
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS or "early stopping" in msg:
        print(msg)

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"restored best model (val {LOSS} {best_val:.6f})")
elif PATIENCE is not None:
    print("PATIENCE set but no validation split (single sample) - early stopping inactive")

if CKPT_PATH is not None:
    os.makedirs(os.path.dirname(CKPT_PATH) or ".", exist_ok=True)
    torch.save({
        "arch": ARCH,
        "in_channels": 3 + len(EXTRA_DIMS),
        "n_points": N_POINTS,
        "latent_dim": LATENT_DIM,
        "output_range": OUTPUT_RANGE,
        "sa_points": list(SA_POINTS),
        "sa_radii": list(SA_RADII),
        "sa_nsample": SA_NSAMPLE,
        "extra_dims": list(EXTRA_DIMS),
        "state_dict": model.state_dict(),
    }, CKPT_PATH)
    print(f"saved checkpoint -> {CKPT_PATH}")

In [ ]:
# ---- reconstruct a train sample and (if available) a val sample ------------
model.eval()


def recon_metrics(x):
    with torch.no_grad():
        xd = x[None].to(device)  # (1, C, N); eval mode so BatchNorm is fine at B=1
        recon, z = model(xd)
        obj = criterion(recon, xd).item()
        mse = nn.functional.mse_loss(recon, xd).item()
    return xd, recon, z, obj, mse


def plot_recon(x, recon, title):
    o = x[0, :3].cpu().numpy()        # original xyz points
    r = recon[0, :3].cpu().numpy()    # reconstructed xyz points
    pairs = [(0, 1, "x", "y"), (0, 2, "x", "z"), (1, 2, "y", "z")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (i, j, li, lj) in zip(axes, pairs):
        ax.scatter(o[i], o[j], s=4, label="original")
        ax.scatter(r[i], r[j], s=4, label="reconstructed")
        ax.set_xlabel(li)
        ax.set_ylabel(lj)
        ax.legend()
    fig.suptitle(title)
    plt.show()


x, recon, z, obj, mse = recon_metrics(dataset[0])
print(f"train sample {dataset.names[0]}: {LOSS} {obj:.6f}, MSE {mse:.6f}, latent {tuple(z.shape)}")
# the scatter series below are in this same model space - if the plot axes
# fall far outside this range, something is wrong; otherwise it's model fit
print(f"input value range: [{float(x.min()):.4g}, {float(x.max()):.4g}] (model space)")
print(f"recon xyz range: [{float(recon[0, :3].min()):.4g}, {float(recon[0, :3].max()):.4g}] "
      f"(bounded by OUTPUT_RANGE={OUTPUT_RANGE!r})")

if n_val:
    xv, recon_v, zv, obj_v, mse_v = recon_metrics(val_set[0])
    name_v = dataset.names[val_set.indices[0]]
    print(f"val sample {name_v}: {LOSS} {obj_v:.6f}, MSE {mse_v:.6f}, latent {tuple(zv.shape)}")
    print(f"recon xyz range: [{float(recon_v[0, :3].min()):.4g}, {float(recon_v[0, :3].max()):.4g}] "
          f"(bounded by OUTPUT_RANGE={OUTPUT_RANGE!r})")
else:
    print("no validation split (single sample) - skipping val reconstruction")

try:
    import matplotlib.pyplot as plt
    plot_recon(x, recon, f"train reconstruction of {dataset.names[0]}")
    if n_val:
        plot_recon(xv, recon_v, f"val reconstruction of {name_v}")
except ImportError:
    print("matplotlib not installed - skipping plots")
